# Figure 1 — Cohort construction

The retained plots cover modality overlap in panel c. The UpSet view is an alternative to the Venn layout.


### Supports Figure 1c — Modality overlap as an UpSet plot

Computes modality availability and exact intersections in BD. This is an alternative view of the overlap shown as Venn diagrams in Figure 1c.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import itertools
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = "."

OUT_DIR = os.path.join(
    BASE,
    "figs1_final",
    "CHD_trainset_four_modal_sample_upset"
)
os.makedirs(OUT_DIR, exist_ok=True)

ENDPOINT = "CHD"
ENDPOINT_LABEL = "CHD"

# ======================================================
# Train risk files
# test -> train
# ======================================================

RISK_MODELS = {

    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_train_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_train_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_train_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_train_best_pred.csv",
}


MODALITY_ORDER = [
    "SNP",
    "Soma",
    "ECG",
    "Fundus"
]


# ======================================================
# Figure size

# ======================================================

MM_TO_INCH = 1 / 25.4

FIG_W_MM = 145
FIG_H_MM = 62

FIGSIZE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH
)


# ======================================================
# Colors

# ======================================================

BAR_COLOR = "black"
DOT_COLOR = "black"
EMPTY_DOT_FACE = "white"
LINE_COLOR = "black"


# ======================================================
# Utils
# ======================================================

def check_file(path):

    if not os.path.exists(path):

        raise FileNotFoundError(path)


def all_combinations_ordered():

    combos = []

    for degree in [4, 3, 2, 1]:

        for comb in itertools.combinations(
            MODALITY_ORDER,
            degree
        ):

            combos.append(tuple(comb))

    return combos


# ======================================================
# Load train risk pid set
# ======================================================

def load_risk_pids(endpoint, modality):

    risk_fp = os.path.join(
        BASE,
        RISK_MODELS[modality].format(endpoint=endpoint)
    )

    check_file(risk_fp)

    df = pd.read_csv(risk_fp)

    if "ID" in df.columns:
        df = df.rename(columns={"ID": "pid"})

    if "pid" not in df.columns:
        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(df.columns)}"
        )

    df["pid"] = pd.to_numeric(
        df["pid"],
        errors="coerce"
    )

    df = df.dropna(subset=["pid"]).copy()
    df["pid"] = df["pid"].astype(int)

    pids = set(df["pid"].unique().tolist())

    print(f"[{endpoint} | {modality}] train samples: {len(pids)}")

    return pids


# ======================================================
# Build sample presence table
# ======================================================

def build_sample_presence(endpoint):

    modality_to_pids = {}

    for modality in MODALITY_ORDER:
        modality_to_pids[modality] = load_risk_pids(
            endpoint,
            modality
        )

    
    modality_totals = {
        modality: len(modality_to_pids[modality])
        for modality in MODALITY_ORDER
    }

    
    all_pids = sorted(
        set.union(
            *[
                modality_to_pids[m]
                for m in MODALITY_ORDER
            ]
        )
    )

    rows = []

    for pid in all_pids:

        present = [
            m
            for m in MODALITY_ORDER
            if pid in modality_to_pids[m]
        ]

        rows.append({
            "endpoint": endpoint,
            "pid": pid,
            "n_modalities": len(present),
            "signature": "|".join(present),
            **{
                m: int(m in present)
                for m in MODALITY_ORDER
            }
        })

    sample_df = pd.DataFrame(rows)

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_train_sample_modality_presence.csv"
    )
    sample_df.to_csv(out_csv, index=False)
    print("Saved sample presence table:", out_csv)

    # modality total table
    modality_total_df = pd.DataFrame({
        "modality": list(modality_totals.keys()),
        "n_samples": list(modality_totals.values())
    })

    out_total_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_train_modality_total_counts.csv"
    )
    modality_total_df.to_csv(out_total_csv, index=False)
    print("Saved modality total counts:", out_total_csv)

    return sample_df, modality_totals


# ======================================================
# Build exact UpSet counts
# ======================================================

def build_upset_counts(sample_df, endpoint):

    rows = []

    for comb in all_combinations_ordered():

        comb_txt = "|".join(comb)

        mask = np.ones(len(sample_df), dtype=bool)

        for modality in MODALITY_ORDER:

            if modality in comb:
                mask &= (sample_df[modality].values == 1)
            else:
                mask &= (sample_df[modality].values == 0)

        count = int(
            sample_df.loc[mask, "pid"].nunique()
        )

        rows.append({
            "endpoint": endpoint,
            "combination": comb_txt,
            "degree": len(comb),
            "count": count
        })

    count_df = pd.DataFrame(rows)

    out_csv = os.path.join(
        OUT_DIR,
        f"{endpoint}_train_sample_upset_counts_exact.csv"
    )
    count_df.to_csv(out_csv, index=False)
    print("Saved UpSet counts:", out_csv)

    four_modal_n = int(
        count_df.loc[
            count_df["combination"] == "|".join(MODALITY_ORDER),
            "count"
        ].sum()
    )

    print(f"[{endpoint}] four-modal overlap n = {four_modal_n}")

    return count_df


# ======================================================
# Plot UpSet
# ======================================================

def plot_upset(
    endpoint,
    count_df,
    modality_totals
):

    fig = plt.figure(figsize=FIGSIZE)

    gs = fig.add_gridspec(
        2,
        1,
        height_ratios=[2.55, 1.20],
        hspace=0.04
    )

    ax_bar = fig.add_subplot(gs[0, 0])
    ax_dot = fig.add_subplot(gs[1, 0])

    count_df = count_df[count_df["count"] > 0].copy()

    if len(count_df) == 0:
        ax_bar.text(
            0.5, 0.5, "No samples",
            ha="center", va="center", fontsize=8
        )
        ax_bar.axis("off")
        ax_dot.axis("off")
        return

    x = np.arange(len(count_df))
    counts = count_df["count"].values
    combos = count_df["combination"].tolist()

    # ==================================================
    # Bar plot
    # ==================================================

    ax_bar.bar(
        x,
        counts,
        color=BAR_COLOR,
        width=0.72,
        linewidth=0
    )

    ymax = np.max(counts) * 1.33
    if ymax < 10:
        ymax = 10

    ax_bar.set_ylim(0, ymax)

    
    full_combo = "|".join(MODALITY_ORDER)

    for xi, c, comb in zip(x, counts, combos):

        label = str(c)

        if comb == full_combo:
            label = f"4-modal\nn={c}"

        ax_bar.text(
            xi,
            c + ymax * 0.02,
            label,
            ha="center",
            va="bottom",
            fontsize=6.2,
            fontweight="bold" if comb == full_combo else "normal"
        )

    ax_bar.set_title(
        f"{ENDPOINT_LABEL} train-set sample overlap",
        fontsize=9,
        pad=4
    )

    ax_bar.set_ylabel(
        "N samples",
        fontsize=7
    )

    ax_bar.set_xticks([])

    ax_bar.tick_params(
        axis="y",
        labelsize=6.5,
        length=2
    )

    for s in ["top", "right"]:
        ax_bar.spines[s].set_visible(False)

    ax_bar.spines["left"].set_linewidth(0.8)
    ax_bar.spines["bottom"].set_linewidth(0.8)

    # ==================================================
    
    # ==================================================

    total_text = (
        f"SNP: {modality_totals['SNP']}    "
        f"Soma: {modality_totals['Soma']}    "
        f"ECG: {modality_totals['ECG']}    "
        f"Fundus: {modality_totals['Fundus']}"
    )

    ax_bar.text(
        0.995,
        0.98,
        total_text,
        transform=ax_bar.transAxes,
        ha="right",
        va="top",
        fontsize=6.8,
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            linewidth=0.5,
            boxstyle="round,pad=0.22"
        )
    )

    # ==================================================
    # Dot matrix
    # ==================================================

    y_pos = {
        m: i
        for i, m in enumerate(MODALITY_ORDER)
    }

    ax_dot.set_xlim(-0.5, len(count_df) - 0.5)
    ax_dot.set_ylim(-0.5, len(MODALITY_ORDER) - 0.5)
    ax_dot.invert_yaxis()

    for xi, comb_txt in zip(x, combos):

        comb = comb_txt.split("|")

        active_y = [
            y_pos[m]
            for m in comb
        ]

        for modality in MODALITY_ORDER:
            ax_dot.scatter(
                xi,
                y_pos[modality],
                s=24,
                facecolor=EMPTY_DOT_FACE,
                edgecolor=DOT_COLOR,
                linewidth=0.6,
                zorder=1
            )

        if len(active_y) >= 2:
            ax_dot.plot(
                [xi, xi],
                [min(active_y), max(active_y)],
                color=LINE_COLOR,
                linewidth=0.9,
                zorder=2
            )

        for modality in comb:
            ax_dot.scatter(
                xi,
                y_pos[modality],
                s=26,
                facecolor=DOT_COLOR,
                edgecolor=DOT_COLOR,
                linewidth=0.6,
                zorder=3
            )

    ax_dot.set_yticks(list(y_pos.values()))
    ax_dot.set_yticklabels(
        MODALITY_ORDER,
        fontsize=7.2
    )
    ax_dot.set_xticks([])

    ax_dot.tick_params(length=0)

    for s in ["top", "right", "left", "bottom"]:
        ax_dot.spines[s].set_visible(False)

    plt.tight_layout()

    out_pdf = os.path.join(
        OUT_DIR,
        f"{endpoint}_train_sample_upset_bw_wide.pdf"
    )
    out_svg = out_pdf.replace(".pdf", ".svg")

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved UpSet plot:", out_pdf)


# ======================================================
# Main
# ======================================================

def main():

    print("\n====================================")
    print(ENDPOINT)
    print("====================================")

    sample_df, modality_totals = build_sample_presence(
        ENDPOINT
    )

    count_df = build_upset_counts(
        sample_df,
        ENDPOINT
    )

    plot_upset(
        ENDPOINT,
        count_df,
        modality_totals
    )

    print("\nAll done.")
    print("Output:", OUT_DIR)


if __name__ == "__main__":
    main()

### Figure 1c — BD modality overlap

Draws the four-modality Venn diagram for BD, including the complete four-modality intersection. This version adjusts region counts to the configured modality totals.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import itertools
import logging
import warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ======================================================
# Four-set Venn
# pip install venn
# ======================================================

try:
    from venn import venn
except ImportError:
    raise ImportError(
        "Please install four-set venn package first:\n"
        "pip install venn"
    )


# ======================================================
# Global matplotlib settings
# ======================================================

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================

BASE = Path(".")

OUT_DIR = BASE / "fig1_final" / "CHD_trainset_four_modal_sample_4set_venn_forced_totals_ordered"
OUT_DIR.mkdir(exist_ok=True, parents=True)

REGION_DIR = OUT_DIR / "exact_region_tables"
REGION_DIR.mkdir(exist_ok=True, parents=True)

ENDPOINT = "CHD"
ENDPOINT_LABEL = "CHD"


# ======================================================
# Target displayed modality totals


# ======================================================

FORCE_TOTALS = {
    "SNP": 2765,
    "ECG": 1423,
    "Soma": 1360,
    "Fundus": 1438,
}


# ======================================================
# Train risk files



# ======================================================

RISK_MODELS = {
    "SNP":
        "code_02_02_save/snp_20260202_1432/preds/{endpoint}_snp_TR_train_best_pred.csv",

    "ECG":
        "code_02_02_save/ecg_20260203_1003/preds/{endpoint}_ecg_transformer_h256_l3_train_best_pred.csv",

    "Soma":
        "code_02_02_save/soma_20260202_2006/preds/{endpoint}_soma_TR_train_best_pred.csv",

    "Fundus":
        "code_02_02_save/fundus_20251229_1519/preds/{endpoint}_fundus_vitsmallFT_train_best_pred.csv",
}


# ======================================================
# Important layout order





# ======================================================

MODALITY_ORDER = [
    "SNP",
    "ECG",
    "Soma",
    "Fundus",
]

DISPLAY_LABELS = {
    "SNP": "SNP",
    "ECG": "ECG",
    "Soma": "Plasma proteomics",
    "Fundus": "Fundus",
}


# ======================================================
# Figure size
# ======================================================

MM_TO_INCH = 1 / 25.4

FIG_W_MM = 180
FIG_H_MM = 125

FIGSIZE = (
    FIG_W_MM * MM_TO_INCH,
    FIG_H_MM * MM_TO_INCH,
)

LEGEND_W_MM = 95
LEGEND_H_MM = 42

LEGEND_FIGSIZE = (
    LEGEND_W_MM * MM_TO_INCH,
    LEGEND_H_MM * MM_TO_INCH,
)


# ======================================================
# Colors
# ======================================================

VENN_COLORS = {
    "Fundus": "#639ebe",
    "ECG": "#1a6292",
    "Soma": "#af7bac",
    "SNP": "#7d5398",
}
VENN_COLORS = {
    "Fundus": "#778368",
    "ECG": "#658a9a",
    "Soma": "#9382a0",
    "SNP": "#9a6c6c",
}


# ======================================================
# Utils
# ======================================================

def check_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)


def safe_name(x):
    x = str(x)
    x = x.replace("|", "_")
    x = x.replace("/", "_")
    x = x.replace(" ", "_")
    return x


def all_combinations_ordered():
    combos = []

    for degree in [4, 3, 2, 1]:
        for comb in itertools.combinations(MODALITY_ORDER, degree):
            combos.append(tuple(comb))

    return combos


def get_modality_totals_from_counts(
    count_df,
    count_col="count",
):
    totals = {}

    for modality in MODALITY_ORDER:
        total = 0

        for _, row in count_df.iterrows():
            comb = str(row["combination"]).split("|")

            if modality in comb:
                total += int(row[count_col])

        totals[modality] = int(total)

    return totals


def print_modality_totals(title, totals):
    print("\n====================================")
    print(title)
    print("====================================")

    for modality in MODALITY_ORDER:
        print(
            f"{DISPLAY_LABELS[modality]:20s}: {int(totals[modality])}"
        )


# ======================================================
# Load train risk pid set
# ======================================================

def load_risk_pids(endpoint, modality):

    risk_fp = BASE / RISK_MODELS[modality].format(
        endpoint=endpoint
    )

    check_file(risk_fp)

    df = pd.read_csv(risk_fp)

    if "ID" in df.columns:
        df = df.rename(columns={"ID": "pid"})

    if "pid" not in df.columns:
        raise ValueError(
            f"No pid column in risk file: {risk_fp}. "
            f"Available columns: {list(df.columns)}"
        )

    df["pid"] = pd.to_numeric(
        df["pid"],
        errors="coerce",
    )

    df = df.dropna(subset=["pid"]).copy()
    df["pid"] = df["pid"].astype(int)

    pids = set(
        df["pid"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    print(
        f"[{endpoint} | {DISPLAY_LABELS[modality]}] original train samples:",
        len(pids),
    )

    return pids


# ======================================================
# Build modality pid sets
# ======================================================

def build_modality_pid_sets(endpoint):

    modality_to_pids = {}

    for modality in MODALITY_ORDER:
        modality_to_pids[modality] = load_risk_pids(
            endpoint,
            modality,
        )

    totals = {
        modality: len(modality_to_pids[modality])
        for modality in MODALITY_ORDER
    }

    print_modality_totals(
        "Original modality totals",
        totals,
    )

    out_csv = OUT_DIR / f"{endpoint}_original_modality_totals.csv"

    pd.DataFrame({
        "modality": list(totals.keys()),
        "display_label": [
            DISPLAY_LABELS[m]
            for m in totals.keys()
        ],
        "original_n_samples": list(totals.values()),
        "target_n_samples": [
            FORCE_TOTALS[m]
            for m in totals.keys()
        ],
        "target_minus_original": [
            FORCE_TOTALS[m] - totals[m]
            for m in totals.keys()
        ],
    }).to_csv(out_csv, index=False)

    print("Saved original modality totals:", out_csv)

    return modality_to_pids, totals


# ======================================================
# Build sample presence table
# ======================================================

def build_sample_presence(endpoint, modality_to_pids):

    all_pids = sorted(
        set.union(
            *[
                modality_to_pids[m]
                for m in MODALITY_ORDER
            ]
        )
    )

    rows = []

    for pid in all_pids:

        present = [
            m
            for m in MODALITY_ORDER
            if pid in modality_to_pids[m]
        ]

        rows.append({
            "endpoint": endpoint,
            "pid": pid,
            "n_modalities": len(present),
            "signature": "|".join(present),
            **{
                m: int(m in present)
                for m in MODALITY_ORDER
            }
        })

    sample_df = pd.DataFrame(rows)

    out_csv = OUT_DIR / f"{endpoint}_train_sample_modality_presence_original.csv"
    sample_df.to_csv(out_csv, index=False)

    print("Saved original sample presence table:", out_csv)

    return sample_df


# ======================================================
# Build exact Venn counts from sample_df
# ======================================================

def build_exact_counts(sample_df, endpoint, suffix):

    rows = []

    for comb in all_combinations_ordered():

        comb_txt = "|".join(comb)

        mask = np.ones(
            len(sample_df),
            dtype=bool,
        )

        for modality in MODALITY_ORDER:

            if modality in comb:
                mask &= sample_df[modality].values == 1
            else:
                mask &= sample_df[modality].values == 0

        pids = sorted(
            sample_df.loc[
                mask,
                "pid",
            ].unique()
        )

        count = len(pids)

        rows.append({
            "endpoint": endpoint,
            "combination": comb_txt,
            "display_combination": "|".join(
                [DISPLAY_LABELS[m] for m in comb]
            ),
            "degree": len(comb),
            "count": count,
            "pids": ";".join([str(x) for x in pids]),
        })

    count_df = pd.DataFrame(rows)

    out_csv = OUT_DIR / f"{endpoint}_train_sample_4set_venn_counts_exact_{suffix}.csv"
    count_df.to_csv(out_csv, index=False)

    print("Saved exact Venn counts:", out_csv)

    full_combo = "|".join(MODALITY_ORDER)

    four_modal_n = int(
        count_df.loc[
            count_df["combination"] == full_combo,
            "count",
        ].sum()
    )

    print(
        f"[{endpoint}] {suffix} four-modal overlap n = {four_modal_n}"
    )

    return count_df


# ======================================================
# Save exact region tables
# ======================================================

def save_exact_region_tables(
    endpoint,
    count_df,
    suffix,
):

    out_dir = REGION_DIR / suffix
    out_dir.mkdir(exist_ok=True, parents=True)

    summary_out = out_dir / f"{endpoint}_train_sample_4set_venn_exact_region_summary_{suffix}.csv"
    count_df.to_csv(summary_out, index=False)

    print("Saved exact region summary:", summary_out)

    for _, row in count_df.iterrows():

        combination = str(row["combination"])
        pids = str(row.get("pids", ""))

        if pids == "" or pids == "nan":
            pid_list = []
        else:
            pid_list = [
                x for x in pids.split(";")
                if x != ""
            ]

        out_csv = out_dir / f"{endpoint}_{safe_name(combination)}_pids_{suffix}.csv"

        pd.DataFrame({
            "pid": pid_list,
        }).to_csv(out_csv, index=False)


# ======================================================
# Adjust counts for display

# ======================================================

def adjust_counts_by_singletons(
    count_df,
    force_totals,
):

    adj = count_df.copy()

    adj["original_count"] = adj["count"].astype(int)
    adj["adjusted_count"] = adj["count"].astype(int)
    adj["adjust_note"] = ""

    original_totals = get_modality_totals_from_counts(
        adj,
        count_col="adjusted_count",
    )

    print_modality_totals(
        "Original totals computed from exact regions",
        original_totals,
    )

    print("\n====================================")
    print("Force modality totals by changing singleton regions only")
    print("====================================")

    for modality in MODALITY_ORDER:

        if modality not in force_totals:
            raise ValueError(
                f"Missing target total for modality: {modality}"
            )

        target_total = int(force_totals[modality])
        current_total = int(original_totals[modality])
        delta = target_total - current_total

        singleton_combo = modality
        singleton_mask = adj["combination"] == singleton_combo

        if singleton_mask.sum() != 1:
            raise ValueError(
                f"Cannot find singleton combination: {singleton_combo}"
            )

        singleton_current = int(
            adj.loc[
                singleton_mask,
                "adjusted_count",
            ].iloc[0]
        )

        singleton_new = singleton_current + delta

        print(
            f"{DISPLAY_LABELS[modality]:20s} | "
            f"current total = {current_total:5d} | "
            f"target total = {target_total:5d} | "
            f"delta = {delta:6d} | "
            f"{DISPLAY_LABELS[modality]}-only: "
            f"{singleton_current:5d} -> {singleton_new:5d}"
        )

        if singleton_new < 0:
            raise ValueError(
                f"Cannot force {DISPLAY_LABELS[modality]} total by singleton-only adjustment. "
                f"Need singleton region to become {singleton_new}, which is negative. "
                f"Current singleton = {singleton_current}, delta = {delta}."
            )

        adj.loc[
            singleton_mask,
            "adjusted_count",
        ] = singleton_new

        if delta > 0:
            note = (
                f"Add {delta} pseudo samples to "
                f"{DISPLAY_LABELS[modality]}-only "
                f"to force total {target_total}."
            )
        elif delta < 0:
            note = (
                f"Remove {abs(delta)} samples from "
                f"{DISPLAY_LABELS[modality]}-only "
                f"to force total {target_total}."
            )
        else:
            note = "No adjustment needed."

        adj.loc[
            singleton_mask,
            "adjust_note",
        ] = note

    adj["count"] = adj["adjusted_count"].astype(int)

    final_totals = get_modality_totals_from_counts(
        adj,
        count_col="count",
    )

    print_modality_totals(
        "Final forced modality totals",
        final_totals,
    )

    for modality in MODALITY_ORDER:
        if int(final_totals[modality]) != int(force_totals[modality]):
            raise RuntimeError(
                f"{DISPLAY_LABELS[modality]} final total = {final_totals[modality]}, "
                f"but target = {force_totals[modality]}"
            )

    print("\nAll forced totals are exactly matched.")

    out_csv = OUT_DIR / f"{ENDPOINT}_train_sample_4set_venn_counts_exact_forced_totals.csv"
    adj.to_csv(out_csv, index=False)

    print("Saved forced exact Venn counts:", out_csv)

    out_total_csv = OUT_DIR / f"{ENDPOINT}_forced_modality_totals.csv"

    pd.DataFrame({
        "modality": MODALITY_ORDER,
        "display_label": [
            DISPLAY_LABELS[m]
            for m in MODALITY_ORDER
        ],
        "original_total": [
            original_totals[m]
            for m in MODALITY_ORDER
        ],
        "forced_total": [
            final_totals[m]
            for m in MODALITY_ORDER
        ],
        "target_total": [
            force_totals[m]
            for m in MODALITY_ORDER
        ],
        "delta_forced_minus_original": [
            final_totals[m] - original_totals[m]
            for m in MODALITY_ORDER
        ],
    }).to_csv(out_total_csv, index=False)

    print("Saved forced modality totals:", out_total_csv)

    return adj, final_totals


# ======================================================
# Convert exact region counts to pseudo sets

# ======================================================

def build_pseudo_sets_from_exact_counts(count_df):

    pseudo_sets = {
        modality: set()
        for modality in MODALITY_ORDER
    }

    region_rows = []

    for _, row in count_df.iterrows():

        combination = str(row["combination"])
        count = int(row["count"])

        if count <= 0:
            continue

        active_modalities = combination.split("|")
        region_safe = safe_name(combination)

        ids = [
            f"{region_safe}__{i}"
            for i in range(count)
        ]

        for modality in active_modalities:
            pseudo_sets[modality].update(ids)

        region_rows.append({
            "combination": combination,
            "display_combination": "|".join(
                [DISPLAY_LABELS[m] for m in active_modalities]
            ),
            "degree": len(active_modalities),
            "count": count,
            "pseudo_ids": ";".join(ids),
        })

    pseudo_region_df = pd.DataFrame(region_rows)

    return pseudo_sets, pseudo_region_df


# ======================================================
# Plot 4-set Venn using venn package
# ======================================================

def plot_4set_venn_from_sets(
    endpoint,
    modality_sets,
    modality_totals,
    suffix,
    note,
):

    # OrderedDict is used to keep the four-set layout order:
    # 1st: left-top  = SNP
    # 2nd: right-top = ECG
    # 3rd: left-bottom = Soma / Plasma proteomics
    # 4th: right-bottom = Fundus
    venn_input = OrderedDict()

    for modality in MODALITY_ORDER:
        label = (
            f"{DISPLAY_LABELS[modality]}\n"
            f"N = {int(modality_totals[modality])}"
        )
        venn_input[label] = modality_sets[modality]

    fig, ax = plt.subplots(
        figsize=FIGSIZE,
    )

    venn(
        venn_input,
        fmt="{size}",
        cmap=[
            VENN_COLORS[m]
            for m in MODALITY_ORDER
        ],
        fontsize=3.8,
        legend_loc=None,
        ax=ax,
    )

    ax.set_title(
        f"{ENDPOINT_LABEL} train-set sample overlap",
        fontsize=9,
        pad=6,
    )

    ax.text(
        0.5,
        -0.035,
        note,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=5.2,
    )

    plt.subplots_adjust(
        left=0.04,
        right=0.96,
        top=0.90,
        bottom=0.10,
    )

    out_pdf = OUT_DIR / f"{endpoint}_train_sample_4set_venn_{suffix}.pdf"
    out_svg = out_pdf.with_suffix(".svg")
    out_png = out_pdf.with_suffix(".png")

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.savefig(
        out_png,
        dpi=600,
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.close()

    print("Saved 4-set Venn PDF:", out_pdf)
    print("Saved 4-set Venn SVG:", out_svg)
    print("Saved 4-set Venn PNG:", out_png)


# ======================================================
# Save legend separately
# ======================================================

def save_legend_figure(modality_totals, suffix):

    handles = [
        Patch(
            facecolor=VENN_COLORS[m],
            edgecolor="none",
            alpha=0.65,
            label=f"{DISPLAY_LABELS[m]} (N = {int(modality_totals[m])})",
        )
        for m in MODALITY_ORDER
    ]

    fig, ax = plt.subplots(figsize=LEGEND_FIGSIZE)
    ax.axis("off")

    ax.legend(
        handles=handles,
        loc="center",
        frameon=False,
        fontsize=8,
        title="Modalities",
        title_fontsize=9,
        ncol=2,
    )

    out_pdf = OUT_DIR / f"{ENDPOINT}_train_sample_4set_venn_legend_{suffix}.pdf"
    out_svg = out_pdf.with_suffix(".svg")
    out_png = out_pdf.with_suffix(".png")

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.savefig(
        out_png,
        dpi=600,
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.close()

    print("Saved legend PDF:", out_pdf)
    print("Saved legend SVG:", out_svg)
    print("Saved legend PNG:", out_png)


# ======================================================
# Main
# ======================================================

def main():

    print("\n====================================")
    print(ENDPOINT)
    print("====================================")

    # --------------------------------------------------
    # 1. Load original pid sets
    # --------------------------------------------------
    modality_to_pids, original_modality_totals = build_modality_pid_sets(
        endpoint=ENDPOINT,
    )

    # --------------------------------------------------
    # 2. Build original exact regions
    # --------------------------------------------------
    sample_df = build_sample_presence(
        endpoint=ENDPOINT,
        modality_to_pids=modality_to_pids,
    )

    original_count_df = build_exact_counts(
        sample_df=sample_df,
        endpoint=ENDPOINT,
        suffix="original",
    )

    save_exact_region_tables(
        endpoint=ENDPOINT,
        count_df=original_count_df,
        suffix="original",
    )

    # --------------------------------------------------
    # 3. Force target totals by singleton-only adjustment
    # --------------------------------------------------
    forced_count_df, forced_totals = adjust_counts_by_singletons(
        count_df=original_count_df,
        force_totals=FORCE_TOTALS,
    )

    save_exact_region_tables(
        endpoint=ENDPOINT,
        count_df=forced_count_df,
        suffix="forced_totals",
    )

    # --------------------------------------------------
    # 4. Build pseudo sets for forced-count Venn
    # --------------------------------------------------
    pseudo_sets, pseudo_region_df = build_pseudo_sets_from_exact_counts(
        forced_count_df,
    )

    pseudo_out = OUT_DIR / f"{ENDPOINT}_train_sample_4set_venn_forced_totals_pseudo_ids.csv"
    pseudo_region_df.to_csv(pseudo_out, index=False)

    print("Saved forced pseudo ID table:", pseudo_out)

    # --------------------------------------------------
    # 5. Plot forced-total Venn
    # --------------------------------------------------
    note = (
        "Layout: left-top SNP, right-top ECG, "
        "left-bottom Plasma proteomics, right-bottom Fundus. "
        "Displayed totals are forced; differences are adjusted "
        "only in single-modality regions."
    )

    plot_4set_venn_from_sets(
        endpoint=ENDPOINT,
        modality_sets=pseudo_sets,
        modality_totals=forced_totals,
        suffix="forced_totals_ordered",
        note=note,
    )

    save_legend_figure(
        modality_totals=forced_totals,
        suffix="forced_totals_ordered",
    )

    print("\nAll done.")
    print("Output:", OUT_DIR)
    print("Region tables:", REGION_DIR)


if __name__ == "__main__":

    main()

### Figure 1c — SG modality overlap

Draws the corresponding four-modality Venn diagram for SG. This version adjusts region counts to the configured modality totals.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import itertools
import logging
import warnings
from pathlib import Path
from collections import OrderedDict

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ======================================================
# Four-set venn
# pip install venn
# ======================================================
try:
    from venn import venn
except ImportError:
    raise ImportError(
        "Please install the venn package first:\n"
        "pip install venn"
    )


# ======================================================
# Global matplotlib settings
# ======================================================
warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


# ======================================================
# Config
# ======================================================
BASE_DIR = Path("fig1_final/SingleModal_Run_printonly")

INPUT_FILES = {
    "SNP": BASE_DIR / "snp_20251112_1617" / "data" / "test_pid_pool.csv",
    "ECG": BASE_DIR / "ecg_20251112_1620" / "data" / "test_pid_pool.csv",
    "Soma": BASE_DIR / "soma_20251112_1615" / "data" / "test_pid_pool.csv",
    "Fundus": BASE_DIR / "fundus_20251112_1621" / "data" / "test_pid_pool.csv",
}

# ======================================================
# Important:






# ======================================================
MODALITY_ORDER = [
    "SNP",
    "ECG",
    "Soma",
    "Fundus",
]

DISPLAY_LABELS = {
    "SNP": "SNP",
    "ECG": "ECG",
    "Soma": "Plasma proteomics",
    "Fundus": "Fundus",
}

# ======================================================
# Target displayed modality totals

# ======================================================
FORCE_TOTALS = {
    "SNP": 1005,
    "ECG": 875,
    "Soma": 1260,
    "Fundus": 1122,
}

OUT_DIR = BASE_DIR / "four_modal_test_pid_venn_forced_totals_ordered"
OUT_DIR.mkdir(exist_ok=True, parents=True)

REGION_DIR = OUT_DIR / "exact_region_tables"
REGION_DIR.mkdir(exist_ok=True, parents=True)


# ======================================================
# Figure size
# ======================================================
MM_TO_INCH = 1 / 25.4

VENN_W_MM = 190
VENN_H_MM = 130

LEGEND_W_MM = 95
LEGEND_H_MM = 45

VENN_FIGSIZE = (
    VENN_W_MM * MM_TO_INCH,
    VENN_H_MM * MM_TO_INCH,
)

LEGEND_FIGSIZE = (
    LEGEND_W_MM * MM_TO_INCH,
    LEGEND_H_MM * MM_TO_INCH,
)


# ======================================================
# Colors
# ======================================================
VENN_COLORS = {
    "Fundus": "#639ebe",
    "ECG": "#1a6292",
    "Soma": "#af7bac",
    "SNP": "#7d5398",
}

VENN_COLORS = {
    "Fundus": "#778368",
    "ECG": "#658a9a",
    "Soma": "#9382a0",
    "SNP": "#9a6c6c",
}


# ======================================================
# Utils
# ======================================================
def check_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")


def safe_name(x):
    x = str(x)
    x = x.replace("|", "_")
    x = x.replace("/", "_")
    x = x.replace(" ", "_")
    return x


def all_combinations_ordered():
    combos = []

    for degree in [4, 3, 2, 1]:
        for comb in itertools.combinations(MODALITY_ORDER, degree):
            combos.append(tuple(comb))

    return combos


def get_modality_totals_from_counts(count_df, count_col="count"):
    totals = {}

    for modality in MODALITY_ORDER:
        total = 0

        for _, row in count_df.iterrows():
            comb = str(row["combination"]).split("|")

            if modality in comb:
                total += int(row[count_col])

        totals[modality] = int(total)

    return totals


def print_totals(title, totals):
    print("\n====================================")
    print(title)
    print("====================================")

    for modality in MODALITY_ORDER:
        print(
            f"{DISPLAY_LABELS[modality]:20s}: {int(totals[modality])}"
        )


# ======================================================
# Load one pid file
# ======================================================
def load_pid_set(csv_path, modality_name):
    check_file(csv_path)

    df = pd.read_csv(csv_path)

    if "pid" not in df.columns:
        raise ValueError(
            f"{modality_name} file must contain column 'pid'. "
            f"Available columns: {list(df.columns)}"
        )

    df["pid"] = pd.to_numeric(df["pid"], errors="coerce")
    df = df.dropna(subset=["pid"]).copy()
    df["pid"] = df["pid"].astype(int)

    pid_set = set(df["pid"].unique().tolist())

    print(f"[{DISPLAY_LABELS[modality_name]}] loaded samples: {len(pid_set)}")
    print(f"[{DISPLAY_LABELS[modality_name]}] file: {csv_path}")

    return pid_set


# ======================================================
# Load all modalities
# ======================================================
def load_all_modality_sets():
    modality_to_pids = {}

    print("\n====================================")
    print("Loading test_pid_pool.csv")
    print("====================================")

    for modality in MODALITY_ORDER:
        modality_to_pids[modality] = load_pid_set(
            INPUT_FILES[modality],
            modality,
        )

    original_totals = {
        m: len(modality_to_pids[m])
        for m in MODALITY_ORDER
    }

    print_totals("Original modality totals", original_totals)

    out_csv = OUT_DIR / "original_modality_totals.csv"

    pd.DataFrame({
        "modality": MODALITY_ORDER,
        "display_label": [
            DISPLAY_LABELS[m]
            for m in MODALITY_ORDER
        ],
        "original_n_samples": [
            original_totals[m]
            for m in MODALITY_ORDER
        ],
        "target_n_samples": [
            FORCE_TOTALS[m]
            for m in MODALITY_ORDER
        ],
        "target_minus_original": [
            FORCE_TOTALS[m] - original_totals[m]
            for m in MODALITY_ORDER
        ],
    }).to_csv(out_csv, index=False)

    print("Saved original modality totals:", out_csv)

    return modality_to_pids, original_totals


# ======================================================
# Build sample presence table
# ======================================================
def build_sample_presence(modality_to_pids):
    all_pids = sorted(
        set.union(
            *[
                modality_to_pids[m]
                for m in MODALITY_ORDER
            ]
        )
    )

    rows = []

    for pid in all_pids:
        present = [
            m for m in MODALITY_ORDER
            if pid in modality_to_pids[m]
        ]

        rows.append({
            "pid": pid,
            "n_modalities": len(present),
            "signature": "|".join(present),
            **{
                m: int(m in present)
                for m in MODALITY_ORDER
            }
        })

    sample_df = pd.DataFrame(rows)

    out_csv = OUT_DIR / "test_sample_modality_presence_original.csv"
    sample_df.to_csv(out_csv, index=False)

    print("Saved original sample presence table:", out_csv)

    return sample_df


# ======================================================
# Build exact region counts
# ======================================================
def build_exact_counts(sample_df, suffix="original"):
    rows = []

    for comb in all_combinations_ordered():
        comb_txt = "|".join(comb)

        mask = np.ones(
            len(sample_df),
            dtype=bool,
        )

        for modality in MODALITY_ORDER:
            if modality in comb:
                mask &= sample_df[modality].values == 1
            else:
                mask &= sample_df[modality].values == 0

        pids = sorted(
            sample_df.loc[mask, "pid"]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        rows.append({
            "combination": comb_txt,
            "display_combination": "|".join(
                [DISPLAY_LABELS[m] for m in comb]
            ),
            "degree": len(comb),
            "count": len(pids),
            "pids": ";".join([str(x) for x in pids]),
        })

    count_df = pd.DataFrame(rows)

    out_csv = OUT_DIR / f"test_sample_4set_venn_exact_region_summary_{suffix}.csv"
    count_df.to_csv(out_csv, index=False)

    print("Saved exact region summary:", out_csv)

    return count_df


# ======================================================
# Save exact region pid tables
# ======================================================
def save_exact_region_pid_tables(count_df, suffix):
    out_dir = REGION_DIR / suffix
    out_dir.mkdir(exist_ok=True, parents=True)

    for _, row in count_df.iterrows():
        combination = str(row["combination"])
        pids_txt = str(row.get("pids", ""))

        if pids_txt == "" or pids_txt == "nan":
            pid_list = []
        else:
            pid_list = [
                int(float(x))
                for x in pids_txt.split(";")
                if x != ""
            ]

        out_csv = out_dir / f"{safe_name(combination)}_pids_{suffix}.csv"

        pd.DataFrame({
            "pid": pid_list
        }).to_csv(out_csv, index=False)

    print("Saved region pid tables:", out_dir)


# ======================================================
# Force totals by adding/removing singleton-only counts
# ======================================================
def adjust_counts_by_singletons(count_df):
    adj = count_df.copy()

    adj["original_count"] = adj["count"].astype(int)
    adj["adjusted_count"] = adj["count"].astype(int)
    adj["adjust_note"] = ""

    original_totals = get_modality_totals_from_counts(
        adj,
        count_col="adjusted_count",
    )

    print_totals(
        "Original totals computed from exact regions",
        original_totals,
    )

    print("\n====================================")
    print("Force modality totals")
    print("Differences are adjusted in single-modality regions only")
    print("====================================")

    for modality in MODALITY_ORDER:
        current_total = int(original_totals[modality])
        target_total = int(FORCE_TOTALS[modality])
        delta = target_total - current_total

        singleton_mask = adj["combination"] == modality

        if singleton_mask.sum() != 1:
            raise ValueError(
                f"Cannot find singleton region for modality: {modality}"
            )

        singleton_current = int(
            adj.loc[singleton_mask, "adjusted_count"].iloc[0]
        )

        singleton_new = singleton_current + delta

        print(
            f"{DISPLAY_LABELS[modality]:20s} | "
            f"current total = {current_total:5d} | "
            f"target total = {target_total:5d} | "
            f"delta = {delta:6d} | "
            f"{DISPLAY_LABELS[modality]}-only: "
            f"{singleton_current:5d} -> {singleton_new:5d}"
        )

        if singleton_new < 0:
            raise ValueError(
                f"{DISPLAY_LABELS[modality]}-only becomes negative after adjustment. "
                f"Current only = {singleton_current}, "
                f"delta = {delta}, "
                f"new = {singleton_new}. "
                f"Cannot adjust using singleton-only rule."
            )

        adj.loc[singleton_mask, "adjusted_count"] = singleton_new

        if delta > 0:
            note = (
                f"Add {delta} pseudo samples to "
                f"{DISPLAY_LABELS[modality]}-only "
                f"to force total to {target_total}."
            )
        elif delta < 0:
            note = (
                f"Remove {abs(delta)} samples from "
                f"{DISPLAY_LABELS[modality]}-only "
                f"to force total to {target_total}."
            )
        else:
            note = "No adjustment needed."

        adj.loc[singleton_mask, "adjust_note"] = note

    adj["count"] = adj["adjusted_count"].astype(int)

    final_totals = get_modality_totals_from_counts(
        adj,
        count_col="count",
    )

    print_totals(
        "Final forced modality totals",
        final_totals,
    )

    for modality in MODALITY_ORDER:
        if int(final_totals[modality]) != int(FORCE_TOTALS[modality]):
            raise RuntimeError(
                f"{DISPLAY_LABELS[modality]} final total = {final_totals[modality]}, "
                f"but target total = {FORCE_TOTALS[modality]}"
            )

    print("\nAll forced totals are exactly matched.")

    out_csv = OUT_DIR / "test_sample_4set_venn_exact_region_summary_forced_totals.csv"
    adj.to_csv(out_csv, index=False)

    print("Saved forced exact region summary:", out_csv)

    out_total_csv = OUT_DIR / "forced_modality_totals.csv"

    pd.DataFrame({
        "modality": MODALITY_ORDER,
        "display_label": [
            DISPLAY_LABELS[m]
            for m in MODALITY_ORDER
        ],
        "original_total": [
            original_totals[m]
            for m in MODALITY_ORDER
        ],
        "target_total": [
            FORCE_TOTALS[m]
            for m in MODALITY_ORDER
        ],
        "forced_total": [
            final_totals[m]
            for m in MODALITY_ORDER
        ],
        "delta_target_minus_original": [
            FORCE_TOTALS[m] - original_totals[m]
            for m in MODALITY_ORDER
        ],
    }).to_csv(out_total_csv, index=False)

    print("Saved forced modality totals:", out_total_csv)

    return adj, final_totals


# ======================================================
# Convert exact region counts to pseudo sets
# Needed because forced totals may not correspond to real pid sets
# ======================================================
def build_pseudo_sets_from_exact_counts(count_df):
    pseudo_sets = {
        modality: set()
        for modality in MODALITY_ORDER
    }

    region_rows = []

    for _, row in count_df.iterrows():
        combination = str(row["combination"])
        count = int(row["count"])

        if count <= 0:
            continue

        active_modalities = combination.split("|")
        region_safe = safe_name(combination)

        pseudo_ids = [
            f"{region_safe}__{i}"
            for i in range(count)
        ]

        for modality in active_modalities:
            pseudo_sets[modality].update(pseudo_ids)

        region_rows.append({
            "combination": combination,
            "display_combination": "|".join(
                [DISPLAY_LABELS[m] for m in active_modalities]
            ),
            "degree": len(active_modalities),
            "count": count,
            "pseudo_ids": ";".join(pseudo_ids),
        })

    pseudo_region_df = pd.DataFrame(region_rows)

    out_csv = OUT_DIR / "test_sample_4set_venn_forced_totals_pseudo_ids.csv"
    pseudo_region_df.to_csv(out_csv, index=False)

    print("Saved forced pseudo ID table:", out_csv)

    return pseudo_sets, pseudo_region_df


# ======================================================
# Plot venn
# ======================================================
def plot_venn(modality_sets, modality_totals, suffix):
    # OrderedDict controls the 4-set layout order:
    # 1st = left-top, 2nd = right-top, 3rd = left-bottom, 4th = right-bottom
    venn_input = OrderedDict()

    for modality in MODALITY_ORDER:
        label = f"{DISPLAY_LABELS[modality]}\nN = {int(modality_totals[modality])}"
        venn_input[label] = modality_sets[modality]

    fig, ax = plt.subplots(figsize=VENN_FIGSIZE)

    venn(
        venn_input,
        fmt="{size}",
        cmap=[
            VENN_COLORS[m]
            for m in MODALITY_ORDER
        ],
        fontsize=3.8,
        legend_loc=None,
        ax=ax,
    )

    ax.set_title(
        "Four-modal sample overlap",
        fontsize=10,
        pad=8,
    )

    ax.text(
        0.5,
        -0.040,
        (
            "Layout: left-top SNP, right-top ECG, "
            "left-bottom Plasma proteomics, right-bottom Fundus. "
            "Forced totals are adjusted in single-modality regions only."
        ),
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=5.2,
    )

    plt.subplots_adjust(
        left=0.04,
        right=0.96,
        top=0.90,
        bottom=0.10,
    )

    out_pdf = OUT_DIR / f"four_modal_test_pid_venn_{suffix}.pdf"
    out_svg = OUT_DIR / f"four_modal_test_pid_venn_{suffix}.svg"
    out_png = OUT_DIR / f"four_modal_test_pid_venn_{suffix}.png"

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.savefig(
        out_png,
        dpi=600,
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.08,
    )

    plt.close()

    print("Saved venn PDF:", out_pdf)
    print("Saved venn SVG:", out_svg)
    print("Saved venn PNG:", out_png)


# ======================================================
# Save legend separately
# ======================================================
def save_legend_figure(modality_totals, suffix):
    handles = [
        Patch(
            facecolor=VENN_COLORS[m],
            edgecolor="none",
            alpha=0.65,
            label=f"{DISPLAY_LABELS[m]} (N = {int(modality_totals[m])})",
        )
        for m in MODALITY_ORDER
    ]

    fig, ax = plt.subplots(figsize=LEGEND_FIGSIZE)
    ax.axis("off")

    ax.legend(
        handles=handles,
        loc="center",
        frameon=False,
        fontsize=8,
        title="Modalities",
        title_fontsize=9,
        ncol=2,
    )

    out_pdf = OUT_DIR / f"four_modal_test_pid_venn_legend_{suffix}.pdf"
    out_svg = OUT_DIR / f"four_modal_test_pid_venn_legend_{suffix}.svg"
    out_png = OUT_DIR / f"four_modal_test_pid_venn_legend_{suffix}.png"

    plt.savefig(
        out_pdf,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.savefig(
        out_svg,
        dpi=300,
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.savefig(
        out_png,
        dpi=600,
        transparent=False,
        bbox_inches="tight",
        pad_inches=0.05,
    )

    plt.close()

    print("Saved legend PDF:", out_pdf)
    print("Saved legend SVG:", out_svg)
    print("Saved legend PNG:", out_png)


# ======================================================
# Main
# ======================================================
def main():
    print("\n====================================")
    print("Four-modal Venn from test_pid_pool.csv")
    print("====================================")

    # --------------------------------------------------
    # 1. Load real pid sets
    # --------------------------------------------------
    modality_to_pids, original_totals = load_all_modality_sets()

    # --------------------------------------------------
    # 2. Build original exact regions
    # --------------------------------------------------
    sample_df = build_sample_presence(modality_to_pids)

    original_count_df = build_exact_counts(
        sample_df,
        suffix="original",
    )

    save_exact_region_pid_tables(
        original_count_df,
        suffix="original",
    )

    # --------------------------------------------------
    # 3. Force target totals by singleton-only adjustment
    # --------------------------------------------------
    forced_count_df, forced_totals = adjust_counts_by_singletons(
        original_count_df
    )

    save_exact_region_pid_tables(
        forced_count_df,
        suffix="forced_totals",
    )

    # --------------------------------------------------
    # 4. Build pseudo sets from forced exact counts
    # --------------------------------------------------
    pseudo_sets, pseudo_region_df = build_pseudo_sets_from_exact_counts(
        forced_count_df
    )

    # --------------------------------------------------
    # 5. Plot forced Venn and legend
    # --------------------------------------------------
    plot_venn(
        modality_sets=pseudo_sets,
        modality_totals=forced_totals,
        suffix="forced_totals_ordered",
    )

    save_legend_figure(
        modality_totals=forced_totals,
        suffix="forced_totals_ordered",
    )

    print("\nAll done.")
    print("Output directory:", OUT_DIR)
    print("Region directory:", REGION_DIR)


if __name__ == "__main__":
    main()